# 01_structures_demo — Python Foundations for Data Engineering

**Week 2 · Day 01** — Moving from SQL queries to Python data pipelines.

**How to use this notebook:**
- Press **`Shift + Enter`** to run a code cell and move to the next one.
- Read the `#` comments—each step shows **what changed** in the variable on the right (`→`).


---
## 0. Environment Sanity Check (30 Seconds)

Verify the active Python environment before every run (like picking the right DB in SQL Workbench).


In [12]:
import sys

py_version = sys.version.split()[0]
print(f"Python version: {py_version}")
print(f"Executable: {sys.executable}")


Python version: 3.13.12
Executable: c:\Users\DELL\Desktop\AWS DE\aws-de-keystone-python\.venv\Scripts\python.exe


---
## 1. The 5 Core Primitives & SQL Equivalents

| Python | Example | SQL type |
|--------|---------|----------|
| `int` | `42` | `BIGINT` |
| `float` | `19.99` | `DOUBLE` |
| `str` | `"HYD"` | `VARCHAR` |
| `bool` | `True` / `False` | `BOOLEAN` |
| `None` | `None` | `NULL` |

**Traps:** `x = 10` assigns; `x == 10` compares. Use `is None`, not `== None`.


### Example — Types, f-strings & None checks


In [22]:
order_id = 42
amount = 19.99
city = "HYD"
is_completed = True
discount = None

status = "completed"
log_path = f"s3://aws-de-keystone-datasets/orders/status={status}"
print(f"log_path={log_path}")
print(f"discount is missing: {discount is None}")

units_sold, unit_price = 7, 24.50
has_discount = discount is not None
revenue = units_sold * unit_price
print(f"revenue={revenue:.2f}, has_discount={has_discount}")


log_path=s3://aws-de-keystone-datasets/orders/status=completed
discount is missing: True
revenue=171.50, has_discount=False


---
## 2. The 4 Data Structures (`list`, `dict`, `tuple`, `set`)

| Structure | Example | SQL-style mental model |
|-----------|---------|-------------------------|
| **list** | `[10, 20, 20]` | Many values in order (duplicates allowed) |
| **dict** | `{"order_id": 101, ...}` | One full row as name → value |
| **tuple** | `(101, 3)` | Two columns on one line—**read only** |
| **set** | `{"u1", "u2"}` | `SELECT DISTINCT` (no duplicates) |

Run each cell top to bottom. Every `print` shows the **current value** after the step above (`→`).


### `list` — ordered, mutable batch

Use `[ ]`. Append and edit in place—like collecting many `amount` values from several rows.


In [ ]:
# --- list: day-to-day operations (values shown after each step) ---
batch = [10, 20, 20]
print(f"start           → batch = {batch}")

batch.append(30)          # add one item at the end
print(f"after append    → batch = {batch}")

batch.extend([40, 50])    # add many items at the end (like UNION ALL rows)
print(f"after extend    → batch = {batch}")

batch.insert(1, 15)       # insert at index 1; items at 1+ shift right
print(f"after insert    → batch = {batch}")

last = batch.pop()        # remove & return last item
print(f"after pop       → batch = {batch}, popped value = {last}")

batch.remove(20)          # remove first matching value (only one 20)
print(f"after remove(20)→ batch = {batch}")

batch.sort()              # sort in place (mutates the list)
print(f"after sort      → batch = {batch}")

copy = batch.copy()       # shallow copy — new list, same numbers inside
copy.append(999)
print(f"after copy+edit → batch = {batch}, copy = {copy}")

copy2 = batch      # deep copy — new list, same numbers inside
copy2.append(1000)
print(f"after copy2+edit → batch = {batch}, copy = {copy2}")

# print(f"len(batch)={len(batch)}, count(20)={batch.count(20)}, index(50)={batch.index(50)}") # This command will fail as 50 is popped
print(f"slice [1:3]     → {batch[1:3]}  # new list, batch unchanged")


start           → batch = [10, 20, 20]
after append    → batch = [10, 20, 20, 30]
after extend    → batch = [10, 20, 20, 30, 40, 50]
after insert    → batch = [10, 15, 20, 20, 30, 40, 50]
after pop       → batch = [10, 15, 20, 20, 30, 40], popped value = 50
after remove(20)→ batch = [10, 15, 20, 30, 40]
after sort      → batch = [10, 15, 20, 30, 40]
after copy+edit → batch = [10, 15, 20, 30, 40], copy = [10, 15, 20, 30, 40, 999]
after copy2+edit → batch = [10, 15, 20, 30, 40, 1000], copy = [10, 15, 20, 30, 40, 1000]
slice [1:3]     → [15, 20]  # new list, batch unchanged


In [46]:
a= [10,20,30,40]
print(a[::-1])

[40, 30, 20, 10]


### `dict` — one row / JSON record

Use `{ }` with **keys** (column names) and **values**. One JSON row from `orders`.


In [56]:
# # --- dict: day-to-day operations (values shown after each step) ---
row = {"order_id": 101, "amount": 250.0, "status": "completed"}
print(f"start           → row = {row}")

row["city"] = "HYD"       # add / overwrite a column on this row
row["amount"] = 500
print(f"after row['city']→ row = {row}")

row.update({"amount": 275.0, "currency": "INR"})  # merge keys in place
print(f"after update    → row = {row}")

# print(row["abc"])

discount = row.get("discount", 0.0)   # COALESCE-style; row unchanged
print(f"get missing key → discount = {discount}, row = {row}")

status = row.pop("status")            # remove key and return its value
print(f"after pop       → status = {status}, row = {row}")

row.setdefault("source", "bronze")    # set only if key missing
print(f"after setdefault→ row = {row}")

print(f"keys   → {list(row.keys())}")
print(f"values → {list(row.values())}")
print(f"items  → {list(row.items())}")
print(f"'order_id' in row → {'order_id' in row}")

snapshot = row.copy()
snapshot["order_id"] = 999
print(f"copy+edit       → row = {row}, snapshot = {snapshot}")


start           → row = {'order_id': 101, 'amount': 250.0, 'status': 'completed'}
after row['city']→ row = {'order_id': 101, 'amount': 500, 'status': 'completed', 'city': 'HYD'}
after update    → row = {'order_id': 101, 'amount': 275.0, 'status': 'completed', 'city': 'HYD', 'currency': 'INR'}
get missing key → discount = 0.0, row = {'order_id': 101, 'amount': 275.0, 'status': 'completed', 'city': 'HYD', 'currency': 'INR'}
after pop       → status = completed, row = {'order_id': 101, 'amount': 275.0, 'city': 'HYD', 'currency': 'INR'}
after setdefault→ row = {'order_id': 101, 'amount': 275.0, 'city': 'HYD', 'currency': 'INR', 'source': 'bronze'}
keys   → ['order_id', 'amount', 'city', 'currency', 'source']
values → [101, 275.0, 'HYD', 'INR', 'bronze']
items  → [('order_id', 101), ('amount', 275.0), ('city', 'HYD'), ('currency', 'INR'), ('source', 'bronze')]
'order_id' in row → True
copy+edit       → row = {'order_id': 101, 'amount': 275.0, 'city': 'HYD', 'currency': 'INR', 'source': 'bro

### `tuple` — a small fixed bundle (read-only)

Use `( )` instead of `[ ]`. **Read only—you cannot change values inside** (unlike a list or dict).

Think of one **order line**: `order_id` and `qty` on the same row—like `SELECT order_id, qty FROM order_items WHERE ...` returning exactly two columns.


In [57]:
# --- tuple: read values only (the tuple itself never changes) ---
order_line = (101, 3)                   # (order_id, qty) — one line on an order
print(f"start           → order_line = {order_line}")

order_id = order_line[0]                # first column in the pair
print(f"read [0]        → order_id = {order_id}, order_line still {order_line}")

qty = order_line[1]                     # second column in the pair
print(f"read [1]        → qty = {qty}, order_line still {order_line}")

print(f"how many values → len(order_line) = {len(order_line)}")

order_id, qty = order_line              # unpack into two names; tuple unchanged
print(f"unpack          → order_id={order_id}, qty={qty}, order_line still {order_line}")

order_line[1] = 5                     # TypeError — cannot edit (use a list or dict to update qty)


start           → order_line = (101, 3)
read [0]        → order_id = 101, order_line still (101, 3)
read [1]        → qty = 3, order_line still (101, 3)
how many values → len(order_line) = 2
unpack          → order_id=101, qty=3, order_line still (101, 3)


TypeError: 'tuple' object does not support item assignment

### `set` — DISTINCT, membership, EXCEPT

No duplicates. `u2 in users` is like filtering by `user_id`. `active - churned` is like `EXCEPT`.


In [30]:
# --- set: day-to-day operations (unique members only) ---
users = {"u1", "u2"}
print(f"start           → users = {users}")

users.add("u3")           # add one member
print(f"after add('u3') → users = {users}")

users.add("u2")           # duplicate ignored
print(f"after add('u2') → users = {users}  # still one u2")

users.discard("u99")      # remove if present; no error if missing
users.remove("u1")        # remove or KeyError if missing
print(f"after removes   → users = {users}")

active = {"u1", "u2", "u3"}
churned = {"u3", "u4"}
print(f"union  active|churned   → {active | churned}")      # DISTINCT combine
print(f"intersect active&{{u2,u3}}→ {active & {'u2', 'u3'}}")  # INNER JOIN keys
print(f"except active-churned    → {active - churned}")       # active not in churned

tags = {"raw"}
tags.update({"curated", "raw"})         # merge in place
print(f"after update    → tags = {tags}")

print(f"u2 in active    → {'u2' in active}")


start           → users = {'u2', 'u1'}
after add('u3') → users = {'u2', 'u3', 'u1'}
after add('u2') → users = {'u2', 'u3', 'u1'}  # still one u2
after removes   → users = {'u2', 'u3'}
union  active|churned   → {'u4', 'u3', 'u2', 'u1'}
intersect active&{u2,u3}→ {'u2', 'u3'}
except active-churned    → {'u2', 'u1'}
after update    → tags = {'raw', 'curated'}
u2 in active    → True


---
## 3. Comprehensions (SQL `SELECT ... WHERE` in one line)

| Python | SQL pattern |
|--------|-------------|
| `[x for x in rows if cond]` | `SELECT x FROM rows WHERE cond` |
| `{k: v for ...}` | Lookup map (id → column) |


In [18]:
ORDERS = [
    {"order_id": 10, "amount": 50.0},
    {"order_id": 11, "amount": 150.0},
    {"order_id": 12, "amount": 200.0},
]


### Example — filter (`WHERE amount > 100`)


In [60]:
big_ids = [r["order_id"] for r in ORDERS if r["amount"] > 100]
print(f"orders over 100 → big_ids = {big_ids}")


orders over 100 → big_ids = [11, 12]


### Example — lookup map


In [61]:
amount_by_id = {r["order_id"]: r["amount"] for r in ORDERS if r["amount"] > 100}
print(f"amount_by_id → {amount_by_id}")
print(f"order 11 amount → {amount_by_id[11]}")


amount_by_id → {11: 150.0, 12: 200.0}
order 11 amount → 150.0


### Example — filter amounts


In [62]:
small_amounts = [r["amount"] for r in ORDERS if r["amount"] < 100]
print(f"amounts under 100 → small_amounts = {small_amounts}")


amounts under 100 → small_amounts = [50.0]


In [59]:
orders = [
    {"order_id": 10, "amount": 50.0},
    {"order_id": 11, "amount": 150.0},
    {"order_id": 12, "amount": 200.0},
]

big_ids = []
for row in orders:
    if row["amount"] > 100:
        big_ids.append(row["order_id"])

print(big_ids)

[11, 12]
